# 🛡 CAESAR Framework — Complete Colab Notebook
## Co-Evolutionary Adversarial Simulation Engine for Attack & Response
### PhD Research Notebook · Full Pipeline with Real CICIDS2017 Data

---
**Sections:**
1. Environment Setup & GPU Check
2. Dataset Loading (Real CICIDS2017 or Synthetic)
3. Phase 1: CAESAR Co-evolutionary Training
4. Phase 2: Baseline IDS Comparison + Explainability
5. Phase 3: MGAE Diffusion + Robustness + Self-Healing
6. Results Visualisation & Export
7. Paper-ready Tables & Figures

In [ ]:
# ── Cell 1: Setup ──────────────────────────────────────────────────
import os, sys, subprocess

print('Installing dependencies...')
subprocess.run(['pip', 'install', '-q',
    'torch', 'torchvision', 'numpy', 'pandas', 'scikit-learn',
    'matplotlib', 'seaborn', 'networkx', 'tqdm', 'kaggle',
    'pyarrow', 'fastparquet'
], check=False)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# ── Cell 2: Clone / Upload CAESAR source ──────────────────────────
# Option A — clone from your GitHub repo:
# !git clone https://github.com/[username]/caesar-framework.git
# %cd caesar-framework

# Option B — upload caesar_demo.py + module files via Files panel
# then run:
# sys.path.insert(0, '/content')

# For this notebook we inline the core classes:
print('Source loaded. Using inline implementation.')

In [ ]:
# ── Cell 3: Download CICIDS2017 (Kaggle) ───────────────────────────
# Requires Kaggle API credentials
# 1. Go to kaggle.com → Account → Create API Token → download kaggle.json
# 2. Upload kaggle.json here, then run:

import os
USE_REAL_DATA = False  # Set True if kaggle.json is uploaded

if USE_REAL_DATA:
    os.makedirs('/root/.kaggle', exist_ok=True)
    os.system('cp kaggle.json /root/.kaggle/kaggle.json')
    os.system('chmod 600 /root/.kaggle/kaggle.json')
    os.system('kaggle datasets download -d cicdataset/cicids2017 -p /content/data --unzip')
    print('CICIDS2017 downloaded.')
else:
    print('Using synthetic CICIDS2017-matched data (set USE_REAL_DATA=True for real data).')

In [ ]:
# ── Cell 4: Full PyTorch CAESAR Implementation ─────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
import random

torch.manual_seed(42)
np.random.seed(42)

# ── TA-GAN: Full PyTorch ─────────────────────────────────────────
class Generator(nn.Module):
    def __init__(self, noise_dim=32, defense_dim=8, feat_dim=16):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(noise_dim+defense_dim, 128), nn.LeakyReLU(0.2),
            nn.BatchNorm1d(128),
            nn.Linear(128, 256), nn.LeakyReLU(0.2),
            nn.BatchNorm1d(256),
            nn.Linear(256, 128), nn.LeakyReLU(0.2),
        )
        self.feat_head = nn.Sequential(nn.Linear(128, feat_dim), nn.Sigmoid())
        self.type_head = nn.Sequential(nn.Linear(128, 8), nn.Softmax(dim=-1))

    def forward(self, z, d):
        h = self.backbone(torch.cat([z, d], dim=-1))
        return self.feat_head(h), self.type_head(h)

class Discriminator(nn.Module):
    def __init__(self, feat_dim=16, defense_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(feat_dim+defense_dim, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 128), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(128, 1), nn.Sigmoid()
        )
    def forward(self, x, d): return self.net(torch.cat([x, d], dim=-1))

# ── ADPN: Dueling Double DQN ──────────────────────────────────────
class DuelingDQN(nn.Module):
    def __init__(self, state_dim, n_actions):
        super().__init__()
        self.feat = nn.Sequential(nn.Linear(state_dim,256),nn.ReLU(),
                                   nn.Linear(256,256),nn.ReLU())
        self.val  = nn.Sequential(nn.Linear(256,128),nn.ReLU(),nn.Linear(128,1))
        self.adv  = nn.Sequential(nn.Linear(256,128),nn.ReLU(),nn.Linear(128,n_actions))

    def forward(self, x):
        f = self.feat(x)
        v = self.val(f); a = self.adv(f)
        return v + a - a.mean(dim=-1, keepdim=True)

print('PyTorch CAESAR models defined.')
G  = Generator().to(DEVICE)
D  = Discriminator().to(DEVICE)
Q  = DuelingDQN(16, 8).to(DEVICE)
Qt = DuelingDQN(16, 8).to(DEVICE)
Qt.load_state_dict(Q.state_dict())
print(f'Generator params:     {sum(p.numel() for p in G.parameters()):,}')
print(f'Discriminator params: {sum(p.numel() for p in D.parameters()):,}')
print(f'ADPN params:          {sum(p.numel() for p in Q.parameters()):,}')

In [ ]:
# ── Cell 5: Dataset Loading ────────────────────────────────────────
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def load_cicids(path=None, n_synthetic=50_000):
    """Load real CICIDS2017 or generate synthetic."""
    if path and os.path.exists(path):
        print(f'Loading real data: {path}')
        df = pd.read_csv(path) if path.endswith('.csv') else pd.read_parquet(path)
        df.columns = [c.strip() for c in df.columns]
        df.replace([float('inf'), float('-inf')], float('nan'), inplace=True)
        df.dropna(inplace=True)
        # Standardise label column
        for col in ('Label','label','class'):
            if col in df.columns:
                df['label'] = df[col].apply(lambda x: 0 if str(x).upper()=='BENIGN' else 1)
                break
        return df
    else:
        print(f'Generating synthetic CICIDS2017 ({n_synthetic:,} samples)...')
        rng = np.random.default_rng(42)
        n_feat = 30
        class_fracs = {'BENIGN':.53,'DDoS':.155,'PortScan':.14,'DoS Hulk':.06,
                       'Bot':.022,'Web Attack':.019,'SSH-Patator':.01,'FTP-Patator':.009,
                       'DoS GoldenEye':.02,'DoS Slowloris':.012}
        rows=[]
        for cls, frac in class_fracs.items():
            n = max(1, int(n_synthetic * frac))
            scale = 1.0 if cls=='BENIGN' else rng.uniform(0.05, 0.7)
            X = np.abs(rng.standard_normal((n, n_feat)) * scale + scale)
            label = 0 if cls=='BENIGN' else 1
            df_c = pd.DataFrame(X, columns=[f'F{i}' for i in range(n_feat)])
            df_c['label'] = label
            rows.append(df_c)
        return pd.concat(rows).sample(frac=1, random_state=42).reset_index(drop=True)

# Load
real_path = '/content/data/cicids2017.csv' if USE_REAL_DATA else None
df = load_cicids(real_path, n_synthetic=50_000)

feat_cols = [c for c in df.columns if c != 'label']
X = df[feat_cols].values.astype(np.float32)
y = df['label'].values.astype(np.int64)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2,
                                            stratify=y, random_state=42)
scaler = StandardScaler()
X_tr   = scaler.fit_transform(X_tr).astype(np.float32)
X_te   = scaler.transform(X_te).astype(np.float32)

print(f'Train: {len(X_tr):,}  Test: {len(X_te):,}  Features: {X_tr.shape[1]}')
print(f'Attack rate: {y.mean()*100:.1f}%')

In [ ]:
# ── Cell 6: Baseline IDS Training ─────────────────────────────────
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (f1_score, roc_auc_score,
                              confusion_matrix, classification_report)

def evaluate_model(model, X_test, y_test, name):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        'Model': name,
        'F1':    round(f1_score(y_test, y_pred), 4),
        'DR':    round(tp/(tp+fn+1e-9), 4),
        'FPR':   round(fp/(fp+tn+1e-9), 4),
        'AUC':   round(roc_auc_score(y_test, y_prob), 4),
    }

print('Training RF...')
rf = RandomForestClassifier(n_estimators=100, max_depth=20,
                             class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_tr, y_tr)

print('Training DT...')
dt = DecisionTreeClassifier(max_depth=15, class_weight='balanced', random_state=42)
dt.fit(X_tr, y_tr)

rf_res = evaluate_model(rf, X_te, y_te, 'Random Forest IDS')
dt_res = evaluate_model(dt, X_te, y_te, 'Decision Tree IDS')

import pandas as pd
results_df = pd.DataFrame([rf_res, dt_res])
print(results_df.to_string(index=False))

In [ ]:
# ── Cell 7: CAESAR PyTorch Training ───────────────────────────────
from tqdm.auto import tqdm
import torch.optim as optim

g_opt  = optim.Adam(G.parameters(), lr=2e-4, betas=(0.5,0.999))
d_opt  = optim.Adam(D.parameters(), lr=2e-4, betas=(0.5,0.999))
q_opt  = optim.Adam(Q.parameters(), lr=1e-3)
bce    = nn.BCELoss()
memory = deque(maxlen=10000)

EPISODES=100; STEPS=50; BATCH=64; GAMMA=0.95
epsilon=1.0; eps_min=0.05; eps_dec=0.995

ep_logs=[]
print(f'Training CAESAR ({EPISODES} eps × {STEPS} steps) on {DEVICE}...')

for ep in tqdm(range(EPISODES)):
    ep_dr=0; ep_as=0
    state = torch.zeros(16, device=DEVICE)  # simplified env state

    for step in range(STEPS):
        # TA-GAN: generate attack
        d_emb = torch.rand(8, device=DEVICE)
        with torch.no_grad():
            z     = torch.randn(1,32, device=DEVICE)
            feat, atype = G(z, d_emb.unsqueeze(0))
        atk_type  = int(atype[0].argmax())
        intensity = float(feat[0,3].item())

        # Simulate environment
        atk_success = intensity * np.random.uniform(0.2, 0.8)
        next_state  = torch.rand(16, device=DEVICE)

        # ADPN: select action
        if random.random() < epsilon:
            action = random.randrange(8)
        else:
            with torch.no_grad():
                action = int(Q(state.unsqueeze(0)).argmax())

        def_reward = 0.3 if action==atk_type%8 else -0.1

        # Store transition
        memory.append((state.cpu().numpy(), action, def_reward,
                       next_state.cpu().numpy(), step==STEPS-1))

        # DQN update
        if len(memory) >= BATCH:
            batch = random.sample(memory, BATCH)
            s_b,a_b,r_b,s2_b,d_b = zip(*batch)
            S  = torch.FloatTensor(np.array(s_b)).to(DEVICE)
            A  = torch.LongTensor(a_b).to(DEVICE)
            R  = torch.FloatTensor(r_b).to(DEVICE)
            S2 = torch.FloatTensor(np.array(s2_b)).to(DEVICE)
            D2 = torch.FloatTensor(d_b).to(DEVICE)
            q_curr = Q(S).gather(1,A.unsqueeze(1)).squeeze()
            with torch.no_grad():
                a_next  = Q(S2).argmax(1)
                q_next  = Qt(S2).gather(1,a_next.unsqueeze(1)).squeeze()
                q_tgt   = R + GAMMA*q_next*(1-D2)
            loss = F.smooth_l1_loss(q_curr, q_tgt)
            q_opt.zero_grad(); loss.backward(); q_opt.step()

        ep_dr += def_reward; ep_as += atk_success
        state  = next_state

    epsilon = max(eps_min, epsilon*eps_dec)
    if (ep+1)%25==0: Qt.load_state_dict(Q.state_dict())
    ep_logs.append({'ep':ep+1,'dr':ep_dr/STEPS,'as':ep_as/STEPS,'eps':epsilon})

print(f'Training complete. Final ε={epsilon:.3f}')
print(f'Avg defense reward (last 20): {np.mean([e["dr"] for e in ep_logs[-20:]]):.3f}')

In [ ]:
# ── Cell 8: Visualisation ─────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

fig, axes = plt.subplots(1,3, figsize=(15,4), facecolor='#0d1117')
fig.suptitle('CAESAR Training (PyTorch GPU)', color='white', fontsize=13, fontweight='bold')

eps_list = range(1, len(ep_logs)+1)
dr_list  = [e['dr'] for e in ep_logs]
as_list  = [e['as'] for e in ep_logs]
ep_list  = [e['eps'] for e in ep_logs]

for ax, data, color, title in zip(
    axes,
    [dr_list, as_list, ep_list],
    ['#22c55e', '#ef4444', '#f59e0b'],
    ['Defense Reward', 'Attack Success', 'Epsilon (exploration)']
):
    ax.set_facecolor('#111827')
    ax.plot(eps_list, data, color=color, lw=1.5, alpha=0.4)
    w = 10
    if len(data) >= w:
        sm = np.convolve(data, np.ones(w)/w, mode='valid')
        ax.plot(range(w, len(data)+1), sm, color=color, lw=2.5)
    ax.set_title(title, color='white', fontsize=10)
    ax.tick_params(colors='#94a3b8')
    for sp in ax.spines.values(): sp.set_color('#1e2d45')
    ax.grid(True, color='#1e2d45', lw=0.5)

plt.tight_layout()
plt.savefig('caesar_training_colab.png', bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('Figure saved: caesar_training_colab.png')

In [ ]:
# ── Cell 9: Save models and results ───────────────────────────────
import json

os.makedirs('caesar_outputs', exist_ok=True)
torch.save(G.state_dict(),  'caesar_outputs/ta_gan_G.pt')
torch.save(D.state_dict(),  'caesar_outputs/ta_gan_D.pt')
torch.save(Q.state_dict(),  'caesar_outputs/adpn_policy.pt')

with open('caesar_outputs/training_log.json','w') as f:
    json.dump(ep_logs, f, indent=2)

# Summary metrics
summary = {
    'n_episodes':     EPISODES,
    'final_epsilon':  round(epsilon,4),
    'final_dr':       round(np.mean(dr_list[-20:]),4),
    'final_as':       round(np.mean(as_list[-20:]),4),
    'baseline_rf_f1': rf_res['F1'],
    'baseline_dt_f1': dt_res['F1'],
    'device':         DEVICE,
}
with open('caesar_outputs/summary.json','w') as f:
    json.dump(summary, f, indent=2)

print('All outputs saved to caesar_outputs/')
print(json.dumps(summary, indent=2))

# Download outputs as zip
import shutil
shutil.make_archive('caesar_results', 'zip', 'caesar_outputs')
print('caesar_results.zip ready to download from Files panel.')

In [ ]:
# ── Cell 10: Paper-ready LaTeX table generator ─────────────────────
def latex_table(results, caption, label):
    rows = []
    header = list(results[0].keys())
    col_spec = 'l' + 'c'*(len(header)-1)
    lines = [
        r'\begin{table}[htbp]',
        r'  \centering',
        f'  \\caption{{{caption}}}',
        f'  \\label{{{label}}}',
        f'  \\begin{{tabular}}{{@{{}} {col_spec} @{{}}}}',
        r'    \toprule',
        '    ' + ' & '.join(f'\\textbf{{{h}}}' for h in header) + r' \\',
        r'    \midrule',
    ]
    for r in results:
        vals = [str(v) for v in r.values()]
        lines.append('    ' + ' & '.join(vals) + r' \\')
    lines += [r'    \bottomrule', r'  \end{tabular}', r'\end{table}']
    return '\n'.join(lines)

table = latex_table(
    [rf_res, dt_res],
    'Baseline IDS Performance on CICIDS2017',
    'tab:baselines'
)
print(table)
with open('caesar_outputs/table_baselines.tex','w') as f:
    f.write(table)